# Part A — Problem & Dataset Selection

**Final Project: Philippine Peso Depreciation Early-Warning Model: Predicting the Next-Month PHP/USD Exchange Rate Using Linear Regression and PCA
**

Every good model starts with a clearly defined problem and an honest look at the data before any math happens.

---
**Student name:** _______________________   **Date:** _______________________

---


**What to do:** Choose an open dataset and a target variable to predict. Load it and report its basic shape and structure.

In [ ]:
import numpy as np
import pandas as pd

np.set_printoptions(precision=4, suppress=True)
pd.set_option("display.max_columns", None)

In [ ]:
# Read the monthly worksheet directly from the official BSP Excel workbook.
BSP_URL = "https://www.bsp.gov.ph/statistics/external/pesodollar.xlsx"

monthly_raw = pd.read_excel(
    BSP_URL,
    sheet_name="monthly",
    header=4,
    usecols="A:D"
)

print("Columns in the BSP monthly worksheet:")
print(monthly_raw.columns.tolist())
display(monthly_raw.head(15))

# The workbook columns shown by the source are: Period (year),
# Unnamed: 2 (month), and Average (monthly average exchange rate).
year = pd.to_numeric(monthly_raw["Period"], errors="coerce").ffill()
month_name = monthly_raw["Unnamed: 2"].astype(str).str.strip().str.lower()
rate = pd.to_numeric(monthly_raw["Average"], errors="coerce")

month_numbers = {
    "january": 1, "february": 2, "march": 3, "april": 4,
    "may": 5, "june": 6, "july": 7, "august": 8,
    "september": 9, "october": 10, "november": 11, "december": 12
}
month = month_name.map(month_numbers)

# Combine the filled year and month into proper monthly dates.
dates = pd.to_datetime(
    {"year": year, "month": month, "day": 1},
    errors="coerce"
)

monthly = pd.DataFrame({"Date": dates, "Rate": rate}).dropna()
monthly["Date"] = pd.to_datetime(monthly["Date"]).dt.to_period("M").dt.to_timestamp()
monthly = monthly.sort_values("Date").drop_duplicates("Date", keep="last")
monthly = monthly.set_index("Date")["Rate"].astype(float)
monthly.index = pd.DatetimeIndex(monthly.index)
monthly.index.name = "FeatureMonth"

print("\nMonthly series loaded successfully.")
print("First month:", monthly.index.min().strftime("%Y-%m"))
print("Last month:", monthly.index.max().strftime("%Y-%m"))
print("Available monthly observations:", len(monthly))

In [ ]:
start_date = "1990-01-01"
end_date = "2026-09-01"
expected_months = pd.date_range(start=start_date, end=end_date, freq="MS")

monthly = monthly.loc[start_date:end_date].reindex(expected_months)
monthly.index = pd.DatetimeIndex(monthly.index)
monthly.index.name = "FeatureMonth"

missing_months = monthly[monthly.isna()].index
print("Expected months:", len(expected_months))
print("Missing monthly rates:", len(missing_months))

if len(missing_months) > 0:
    print("Missing months:", missing_months.strftime("%Y-%m").tolist())
    raise ValueError(
        "Some monthly exchange rates are missing. Check the BSP worksheet "
        "before continuing; do not fill missing exchange rates with guesses."
    )

# Construct the three features and the next-month target.
model_data = pd.DataFrame(index=monthly.index)
model_data["CurrentRate"] = monthly
model_data["Trailing3MonthAverage"] = monthly.rolling(window=3).mean()
model_data["YearOverYearChange"] = monthly - monthly.shift(12)
model_data["NextMonthRate"] = monthly.shift(-1)

feature_names = [
    "CurrentRate",
    "Trailing3MonthAverage",
    "YearOverYearChange"
]
target_name = "NextMonthRate"

# Remove rows that do not have all required feature and target values.
model_data = model_data.dropna(subset=feature_names + [target_name])

# Separate the feature matrix (X) and target vector (y).
X_raw = model_data[feature_names].to_numpy(dtype=float)
y = model_data[target_name].to_numpy(dtype=float)

part_a_dataset = model_data.copy()
part_a_dataset["TargetMonth"] = part_a_dataset.index + pd.DateOffset(months=1)

n_samples, n_features = X_raw.shape
print("\nPrepared dataset shape:", part_a_dataset.shape)
print("Number of samples (n):", n_samples)
print("Number of features (p):", n_features)
print("Feature names:", feature_names)
print("Target variable:", target_name)
print("Feature months:", part_a_dataset.index.min().strftime("%Y-%m"),
      "to", part_a_dataset.index.max().strftime("%Y-%m"))
print("Target months:", part_a_dataset["TargetMonth"].min().strftime("%Y-%m"),
      "to", part_a_dataset["TargetMonth"].max().strftime("%Y-%m"))

In [ ]:
print("First five observations:")
display(part_a_dataset.head())

print("\nDescriptive statistics for the three independent features:")
display(pd.DataFrame(X_raw, columns=feature_names).describe())

print("\nDescriptive statistics for the target:")
display(pd.Series(y, name=target_name).describe().to_frame())

print("\nMissing values in features and target:")
display(part_a_dataset[feature_names + [target_name]].isna().sum().to_frame("Missing Values"))

print("Duplicate feature-month dates:", part_a_dataset.index.duplicated().sum())

### Answer these in your report

**Q1.** What real-world quantity are you trying to predict, and why is it continuous rather than categorical?

> *Your answer:* _(write here)_

**Q2.** Where does your dataset come from? Is it open/public? Provide the source (link or citation).

> *Your answer:* _(write here)_

**Q3.** How many samples (n) and how many features (p) does your dataset have?

> *Your answer:* _(write here)_

**Q4.** Before doing any analysis, which features do you expect to be most predictive, and why?

> *Your answer:* _(write here)_

### Save your work for the next notebook

In [ ]:
part_a_dataset.to_csv("peso_usd_part_a_dataset.csv", index_label="FeatureMonth")

%store X_raw
%store y
%store feature_names

print("Saved: peso_usd_part_a_dataset.csv")
print("X_raw shape:", X_raw.shape)
print("y shape:", y.shape)
print("feature_names:", feature_names)